In [1]:
import collections
import collections.abc
collections.Mapping = collections.abc.Mapping
from experta import KnowledgeEngine, Rule, Fact, P

In [ ]:
class HeartDiseaseExpert(KnowledgeEngine):
    def __init__(self):
        super().__init__()
        self.prediction = 0 

    #10 RULES 
    
    # Rule 1: High Blood Pressure in Males
    @Rule(Fact(trestbps=P(lambda x: x >= 140)), Fact(sex=1))
    def rule_1(self): self.prediction = 1

    # Rule 2: High Cholesterol
    @Rule(Fact(chol=P(lambda x: x >= 240)))
    def rule_2(self): self.prediction = 1

    # Rule 3: Chest Pain (Any symptomatic type)
    @Rule(Fact(cp=P(lambda x: x > 0)))
    def rule_3(self): self.prediction = 1

    # Rule 4: Vessel Blockage (ca >= 1)
    @Rule(Fact(ca=P(lambda x: x >= 1)))
    def rule_4(self): self.prediction = 1

    # Rule 5: Exercise Induced Angina
    @Rule(Fact(exang=1))
    def rule_5(self): self.prediction = 1

    # Rule 6: Significant ST Depression (Oldpeak)
    @Rule(Fact(oldpeak=P(lambda x: x >= 2.0)))
    def rule_6(self): self.prediction = 1

    # Rule 7: Age/Sex Risk (Males over 45)
    @Rule(Fact(age=P(lambda x: x > 45)), Fact(sex=1))
    def rule_7(self): self.prediction = 1

    # Rule 8: High Fasting Blood Sugar
    @Rule(Fact(fbs=1))
    def rule_8(self): self.prediction = 1

    # Rule 9: Abnormal Thalassemia
    @Rule(Fact(thal=P(lambda x: x >= 2)))
    def rule_9(self): self.prediction = 1

    # Rule 10: Combined High Risk (BP + CP + Vessel)
    @Rule(Fact(cp=P(lambda x: x > 0)) & 
          Fact(trestbps=P(lambda x: x >= 140)) & 
          Fact(ca=P(lambda x: x >= 1)))
    def rule_10(self): self.prediction = 1

# THE METRIC METHOD
def calculate_knowledge_accuracy(engine, session_data):
    if not session_data:
        return 0.0
    
    correct_count = 0
    for record in session_data:
        engine.reset()
        engine.prediction = 0
        # Send everything except the 'label' to the engine
        features = {k: v for k, v in record.items() if k != 'label'}
        engine.declare(Fact(**features))
        engine.run()
        
        if engine.prediction == record['label']:
            correct_count += 1
            
    return (correct_count / len(session_data)) * 100

# MAIN EXECUTION LOOP
def run_expert_system():
    engine = HeartDiseaseExpert()
    session_records = []  # Stores user inputs to calculate accuracy over time

    print("=== HEART DISEASE EXPERT SYSTEM (CONSOLE MODE) ===")
    
    while True:
        try:
            print("\n--- ENTER PATIENT DATA ---")
            data = {
                "age": float(input("Age: ")),
                "sex": int(input("Sex (1=M, 0=F): ")),
                "trestbps": float(input("Blood Pressure: ")),
                "chol": float(input("Cholesterol: ")),
                "cp": int(input("Chest Pain (0-3): ")),
                "ca": int(input("Vessels (0-3): ")),
                "exang": int(input("Angina (1=Y, 0=N): ")),
                "oldpeak": float(input("Oldpeak: ")),
                "fbs": int(input("FBS > 120 (1=Y, 0=N): ")),
                "thal": int(input("Thal (1, 2, 3): "))
            }
            
            # Since the user is entering data for "testing," we need the ground truth
            actual_label = int(input("Actual Health Status (1=Heart Disease, 0=Healthy): "))
            data['label'] = actual_label
            
            # 1. Store in session for accuracy calculation
            session_records.append(data)

            # 2. Run Single Prediction for current input
            engine.reset()
            engine.prediction = 0
            features_only = {k: v for k, v in data.items() if k != 'label'}
            engine.declare(Fact(**features_only))
            engine.run()
            
            diagnosis = "HIGH RISK" if engine.prediction == 1 else "LOW RISK"
            print(f"\n[ENGINE RESULT]: {diagnosis}")

            # 3. Calculate and Display Knowledge Accuracy for all entered data
            accuracy = calculate_knowledge_accuracy(engine, session_records)
            
            print(f"KNOWLEDGE ACCURACY (Session): {accuracy:.2f}%")
            print(f"Total Patients Evaluated: {len(session_records)}")

            cont = input("\nEvaluate another patient? (y/n): ").lower()
            if cont != 'y':
                break

        except ValueError:
            print("\nERROR: Invalid numeric input. Try again.")

if __name__ == "__main__":
    run_expert_system()

=== HEART DISEASE EXPERT SYSTEM (CONSOLE MODE) ===

--- ENTER PATIENT DATA ---

[ENGINE RESULT]: HIGH RISK
KNOWLEDGE ACCURACY (Session): 100.00%
Total Patients Evaluated: 1


# Comparison: Decision Tree (ML) vs. Expert System (Rules)

This document provides a high-level comparison between Machine Learning Decision Trees and Rule-Based Expert Systems for Heart Disease Detection.

| Metric | Decision Tree (ML) | Expert System (Rules) |
| :--- | :--- | :--- |
| **Accuracy** | Higher (learns from data patterns) | Lower (depends on human knowledge) |
| **Explainability** | Hard to explain deep or complex trees | **Excellent** (Traceable and transparent rules) |
| **Handling Outliers** | Good | **Excellent** (Uses hard clinical thresholds) |
| **Maintenance** | Needs data re-collection and retraining | Simple (Just edit the rule text) |

---
*Note: While Decision Trees offer higher predictive power by uncovering hidden patterns in data, Expert Systems provide the transparency required in clinical settings where doctors need to know 'why' a diagnosis was reached.*